# 2017 修订缺口的原生 Qlib 来源映射修复

源 Recorder、证据哈希、预算与停止条件见 `plan.json`。零市场结果、模型、账户。

In [1]:
import os,json,hashlib,copy,tarfile,tempfile,subprocess,shlex,sys
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.nas_market_data import archive_nas_recorder,_settings
assert 'qlib_venv' in sys.executable,sys.executable
root=Path.cwd();card=root/'research/stage1_soxx_concentration_amendment_gap_resolution_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'source_audit.json').exists() and not (card/'run_state.json').exists()
spec=plan['source_map'];source=root/spec['card'];art=source/'qlib_recorder'/spec['experiment_id']/spec['recorder_id']/'artifacts'
assert sha(source/'summary.json')==spec['summary_sha256'] and sha(source/'source_audit.json')==spec['source_audit_sha256']
for object_name,key in [('mapped_signals','mapped_signals_artifact_sha256'),('amendments','amendments_artifact_sha256'),('source_events','source_events_artifact_sha256')]:
 assert sha(art/'analysis'/object_name)==spec[key]
source_summary=json.loads((source/'summary.json').read_text());assert source_summary['nas_evidence']['sha256']==spec['NAS_archive_sha256'] and source_summary['nas_readback']['verified']
am_spec=plan['amendment_evidence'];am_path=root/am_spec['card']/am_spec['result'];assert sha(am_path)==am_spec['result_sha256']
am_result=json.loads(am_path.read_text());assert am_result['qualified_unchanged_count']==4
assert all(x['numeric_equity_unchanged_after_whitespace_only'] for x in am_result['checks'])
qualified_accessions={x['accession'] for x in am_result['checks']};assert len(qualified_accessions)==4
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['Stage1_train_span'][0],end_time=plan['Stage1_train_span'][1],freq='day'))
assert len(calendar)==plan['expected']['calendar_sessions']
with R.uri_context(spec['recorder_uri']):
 old_rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_id=spec['experiment_id'])
 old=old_rec.load_object('analysis/mapped_signals')
 amendments=old_rec.load_object('analysis/amendments')
 events=old_rec.load_object('analysis/source_events')
assert len(old)==len(calendar)==3020 and len(events)==plan['expected']['base_report_events'] and len(amendments)==5
assert pd.DatetimeIndex(old.signal_date).equals(calendar)
mask=old.amendment_uncertain.astype(bool);uncertain=old.loc[mask]
assert int(mask.sum())==plan['expected']['old_amendment_uncertain_sessions']==11
assert int(old.qualified_source.sum())==plan['expected']['old_qualified_sessions']==3009
assert uncertain.signal_date.min().strftime('%Y-%m-%d')==plan['expected']['uncertain_first_signal']
assert uncertain.signal_date.max().strftime('%Y-%m-%d')==plan['expected']['uncertain_last_signal']
assert set(uncertain.report_period)=={'2017-03-31'} and set(uncertain.accession)=={'0001193125-17-194722'}
assert uncertain.source_known.all() and not uncertain.qualified_source.any()
patched=old.copy(deep=True);patched.loc[mask,'amendment_uncertain']=False;patched.loc[mask,'qualified_source']=True
unchanged=[x for x in old.columns if x not in ('amendment_uncertain','qualified_source')]
pd.testing.assert_frame_equal(old[unchanged],patched[unchanged],check_exact=True)
assert int(patched.amendment_uncertain.sum())==0 and int(patched.qualified_source.sum())==3020
assert patched.source_known.all() and np.isfinite(patched[['top5','HHI','source_age_days']].to_numpy(float)).all()
patched_amendments=copy.deepcopy(amendments)
for x in patched_amendments:
 if x['accession'] in qualified_accessions:
  assert not x['equity_semantics_qualified']
  x['equity_semantics_qualified']=True;x['equity_numeric_unchanged']=True;x['concentration_source_age_reset']=False
assert sum(x['equity_semantics_qualified'] for x in patched_amendments)==5
assert len({x['accession'] for x in patched_amendments})==5
frame=patched.set_index('signal_date')[['top5','HHI','source_known','amendment_uncertain','qualified_source','source_age_days']]
frame.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['top5','HHI']],'meta':frame[['source_known','amendment_uncertain','qualified_source','source_age_days']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'Stage1_train':tuple(plan['Stage1_train_span'])})
prepared=dataset.prepare('Stage1_train',col_set=['feature','meta'])
assert len(prepared)==3020 and int(prepared[('meta','qualified_source')].sum())==3020
assert int(prepared[('meta','amendment_uncertain')].sum())==0
audit={'status':'conservative_filing_date_Stage1_concentration_train_calendar_complete_after_unchanged_amendments',
 'plan_sha256':sha(card/'plan.json'),'old_recorder_id':spec['recorder_id'],'base_report_events':len(events),
 'train_Qlib_sessions':len(calendar),'old_qualified_sessions':int(old.qualified_source.sum()),
 'new_qualified_sessions':int(patched.qualified_source.sum()),'old_amendment_uncertain_sessions':int(mask.sum()),
 'new_amendment_uncertain_sessions':int(patched.amendment_uncertain.sum()),'exact_changed_flag_rows':int(mask.sum()),
 'unchanged_numeric_age_and_original_source_columns':unchanged,
 'first_changed_signal':uncertain.signal_date.min().strftime('%Y-%m-%d'),
 'last_changed_signal':uncertain.signal_date.max().strftime('%Y-%m-%d'),
 'original_amendments':len(amendments),'numeric_unchanged_amendments_newly_qualified':len(qualified_accessions),
 'source_age_reset_from_unchanged_amendments':False,
 'historical_first_public_intraday_receipts_qualified':False,'held_independent':False,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True)
tracking=card/'qlib_recorder'
with R.start(experiment_name='SOXX_original_concentration_Stage1_amendment_resolved',uri=str(tracking)):
 rec=R.get_recorder()
 rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'source_map':spec['mapped_signals_artifact_sha256'],'amendments':am_spec['result_sha256']})
 rec.save_objects(artifact_path='analysis',source_events=events,amendments=patched_amendments,mapped_signals=patched,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_report_events=len(events),Stage1_train_sessions=len(calendar),qualified_source_sessions=3020,amendment_uncertain_sessions=0,model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id)
 rb_map=checked.load_object('analysis/mapped_signals')
 rb_dataset=checked.load_object('metadata/dataset')
 assert len(rb_map)==3020 and rb_map.qualified_source.all() and not rb_map.amendment_uncertain.any()
 assert len(rb_dataset.prepare('Stage1_train',col_set=['feature','meta']))==3020
settings=_settings(root);binding=card/'nas_binding.json'
binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_original_concentration_Stage1_amendment_resolved','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-amendment-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:
  run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as saved:
  names=[m.name for m in saved.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/analysis/mapped_signals') for x in names)
 assert any(x.endswith('/artifacts/metadata/dataset') for x in names)
summary={'experiment':'stage1_soxx_concentration_amendment_gap_resolution_20261023',
 'decision':'conservative_Stage1_train_source_calendar_complete_amendment_gap_closed_information_unrun',
 'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'source_audit.json',
 'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),
 'source_map_old_recorder_id':spec['recorder_id'],'train_sessions':3020,'qualified_source_sessions':3020,
 'new_model_fits':0,'new_native_accounts':0,'source_NAS_evidence':evidence,
 'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':plan['artifact_policy'],
 'limitations':['Conservative filing-date-to-next-Qlib-session calendar only; exact historical first-public intraday clock not certified','All 2023-2026 reused diagnostic and independent forward evidence remain separate','Source completeness and unchanged amendment values do not prove SOXX predictive or fee-net value']}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
(card/'run_state.json').write_text(json.dumps({'status':'completed','recorder_id':recorder_id,'experiment_id':experiment_id,'plan_sha256':sha(card/'plan.json'),'new_model_fits':0,'new_native_accounts':0},indent=2)+'\n')
print({'status':summary['decision'],'qualified_sessions':3020,'changed_flags':11,'recorder_id':recorder_id,'NAS_sha256':evidence['sha256']})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[68174:MainThread](2026-10-23 03:16:53,152) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[68174:MainThread](2026-10-23 03:16:53,153) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[68174:MainThread](2026-10-23 03:16:53,154) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[68174:MainThread](2026-10-23 03:16:53,209) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

[68174:MainThread](2026-10-23 03:16:53,214) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name SOXX_original_concentration_Stage1_amendment_resolved.


[68174:MainThread](2026-10-23 03:16:53,216) INFO - qlib.workflow - [exp.py:258] - Experiment 254587370707838823 starts running ...


[68174:MainThread](2026-10-23 03:16:53,295) INFO - qlib.workflow - [recorder.py:345] - Recorder 479e9d93a77a47198c23a08e429d8598 starts running under Experiment 254587370707838823 ...


[68174:MainThread](2026-10-23 03:16:54,208) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{'status': 'conservative_Stage1_train_source_calendar_complete_amendment_gap_closed_information_unrun', 'qualified_sessions': 3020, 'changed_flags': 11, 'recorder_id': '479e9d93a77a47198c23a08e429d8598', 'NAS_sha256': 'd28046b6f86009838e232fa4cec8ca5657226bdd89abd409b785a8cd11c827e3'}


## 中文结论

原生 Qlib 训练日 3,020/3,020 来源资格闭合，仅修订不确定的 11 日资格标记变更；数值、原始 accession 与来源年龄不变。Recorder 479e9d93a77a47198c23a08e429d8598、NAS SHA d28046b6f86009838e232fa4cec8ca5657226bdd89abd409b785a8cd11c827e3。这不是预测力或 SOXX 超额；复用诊断/独立前向仍待核。
